# Settimana 17 — LLMOps: osservabilità, trace e costi

**Obiettivo:** strumentare ResearchAgent con un sistema di trace proprio, senza dipendenze esterne.
Alla fine avrai una dashboard a riga di comando che mostra costo, latenza e tasso di successo per ogni esecuzione.

**Cosa costruisci** in `proyectos/research-agent/`:
- `observability/tracer.py` — `Tracer` e `Span` per misurare latenza e token
- `observability/cost.py` — calcolatrice di costi multi-modello
- `observability/logger.py` — logger strutturato in JSON
- `observability/dashboard.py` — riepilogo di un'esecuzione a riga di comando
- Strumentazione di `ResearchAgent` per emettere trace

**Prerequisiti:** settimane 15-16 (ResearchAgent + API).

**Costo stimato:** $0.01–0.03 se esegui l'agente strumentato una volta.

## Cella 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tiktoken", "python-dotenv"])

import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("OPENAI_API_KEY"), "Definisci OPENAI_API_KEY nel tuo .env"

PROJECT_DIR = Path("../proyectos/research-agent").resolve()
assert PROJECT_DIR.exists(), "Esegui prima il notebook della settimana 15"

OBS_DIR = PROJECT_DIR / "observability"
OBS_DIR.mkdir(exist_ok=True)
(OBS_DIR / "__init__.py").write_text("")

sys.path.insert(0, str(PROJECT_DIR))
print(f"✓ Progetto: {PROJECT_DIR}")
print(f"✓ Directory observability: {OBS_DIR}")

## Cella 1 — Calcolatrice di costi (`cost.py`)

Inizia dalla cosa più concreta: quanto costa ogni chiamata all'LLM.

In [ ]:
COST_PY = '''
"""Calcolatrice di costi per le chiamate agli LLM."""
import tiktoken

# Prezzi in USD per 1M token (approssimati, giugno 2026)
PRECIOS: dict[str, tuple[float, float]] = {
    "gpt-4o-mini":          (0.15,  0.60),
    "gpt-4o":               (2.50, 10.00),
    "gpt-4o-2024-11-20":    (2.50, 10.00),
    "text-embedding-3-small": (0.02, 0.0),
    "text-embedding-3-large": (0.13, 0.0),
}


def precio_modelo(modelo: str) -> tuple[float, float]:
    """Restituisce (prezzo_input, prezzo_output) in USD/1M token."""
    # Cerca corrispondenza esatta, poi prefisso
    if modelo in PRECIOS:
        return PRECIOS[modelo]
    for clave, precios in PRECIOS.items():
        if modelo.startswith(clave.split("-202")[0]):
            return precios
    return (0.15, 0.60)  # fallback: gpt-4o-mini


def calcular_coste(
    tokens_input: int,
    tokens_output: int,
    modelo: str = "gpt-4o-mini",
) -> float:
    """Calcola il costo in USD di una chiamata all'LLM."""
    p_in, p_out = precio_modelo(modelo)
    return (tokens_input * p_in + tokens_output * p_out) / 1_000_000


def contar_tokens(texto: str, modelo: str = "gpt-4o-mini") -> int:
    """Conta i token di un testo senza chiamare l'API."""
    try:
        enc = tiktoken.encoding_for_model(modelo)
    except KeyError:
        enc = tiktoken.get_encoding("cl100k_base")
    return len(enc.encode(texto))


def coste_mensual_estimado(
    tokens_por_llamada: int,
    llamadas_por_dia: int,
    modelo: str = "gpt-4o-mini",
    ratio_output: float = 0.3,
) -> dict:
    """
    Stima il costo mensile dato un volume d'uso.
    ratio_output: frazione di token che sono output (default 30%).
    """
    t_in  = int(tokens_por_llamada * (1 - ratio_output))
    t_out = int(tokens_por_llamada * ratio_output)
    coste_llamada = calcular_coste(t_in, t_out, modelo)
    coste_dia     = coste_llamada * llamadas_por_dia
    coste_mes     = coste_dia * 30
    return {
        "modelo":            modelo,
        "tokens_por_llamada": tokens_por_llamada,
        "llamadas_por_dia":  llamadas_por_dia,
        "coste_por_llamada_usd": round(coste_llamada, 6),
        "coste_dia_usd":     round(coste_dia, 4),
        "coste_mes_usd":     round(coste_mes, 2),
    }
'''

(OBS_DIR / "cost.py").write_text(COST_PY.strip())
print("✓ observability/cost.py generato")

# Demo
import importlib.util, types

spec = importlib.util.spec_from_file_location("cost", OBS_DIR / "cost.py")
cost = types.ModuleType("cost")
spec.loader.exec_module(cost)

# Costo di un report tipico di ResearchAgent
T_IN, T_OUT = 18_000, 2_000

print("\nCosto di un report tipico (18k input + 2k output token):")
for modelo in ["gpt-4o-mini", "gpt-4o"]:
    c = cost.calcular_coste(T_IN, T_OUT, modelo)
    print(f"  {modelo:<20} ${c:.5f}")

print()
print("Proiezione mensile (10 report/giorno, gpt-4o-mini):")
est = cost.coste_mensual_estimado(
    tokens_por_llamada=20_000, llamadas_por_dia=10, modelo="gpt-4o-mini"
)
for k, v in est.items():
    print(f"  {k}: {v}")

## Cella 2 — Tracer e Span (`tracer.py`)

Il `Tracer` raccoglie tutti gli span di un'esecuzione e calcola il riepilogo alla fine.

In [ ]:
TRACER_PY = '''
"""Sistema di trace per ResearchAgent."""
import time
import uuid
import json
import threading
from dataclasses import dataclass, field
from datetime import datetime
from pathlib import Path
from observability.cost import calcular_coste


@dataclass
class Span:
    nombre:        str
    inicio:        float      = field(default_factory=time.perf_counter)
    fin:           float      = 0.0
    tokens_input:  int        = 0
    tokens_output: int        = 0
    modelo:        str        = "gpt-4o-mini"
    ok:            bool       = True
    error:         str        = ""
    metadatos:     dict       = field(default_factory=dict)

    def cerrar(self, ok: bool = True, error: str = ""):
        self.fin   = time.perf_counter()
        self.ok    = ok
        self.error = error

    @property
    def latencia_ms(self) -> float:
        return (self.fin - self.inicio) * 1000 if self.fin else 0.0

    @property
    def coste_usd(self) -> float:
        return calcular_coste(self.tokens_input, self.tokens_output, self.modelo)


class Tracer:
    """
    Raccoglie gli span di un'esecuzione dell'agente.
    Thread-safe per l'uso in background task.

    Uso:
        tracer = Tracer(tema="IA nell'acquacoltura")
        with tracer.span("planificar") as s:
            resultado = crear_plan(tema)
            s.tokens_input = 500
            s.tokens_output = 200
        tracer.guardar()  # salva la trace in JSON
    """

    def __init__(self, tema: str, trazas_dir: str = "./trazas"):
        self.trace_id   = str(uuid.uuid4())[:8]
        self.tema       = tema
        self.inicio     = time.perf_counter()
        self.spans:  list[Span] = []
        self._lock   = threading.Lock()
        self.trazas_dir = Path(trazas_dir)
        self.trazas_dir.mkdir(parents=True, exist_ok=True)

    class _SpanCtx:
        """Context manager che chiude lo span automaticamente."""
        def __init__(self, tracer: "Tracer", span: Span):
            self._tracer = tracer
            self._span   = span

        def __enter__(self) -> Span:
            return self._span

        def __exit__(self, exc_type, exc_val, exc_tb):
            ok    = exc_type is None
            error = str(exc_val) if exc_val else ""
            self._span.cerrar(ok=ok, error=error)
            with self._tracer._lock:
                self._tracer.spans.append(self._span)
            return False  # non sopprime le eccezioni

    def span(self, nombre: str, modelo: str = "gpt-4o-mini") -> "_SpanCtx":
        """Crea uno span e lo aggiunge alla trace all'uscita dal contesto."""
        return self._SpanCtx(self, Span(nombre=nombre, modelo=modelo))

    @property
    def duracion_total_s(self) -> float:
        return time.perf_counter() - self.inicio

    @property
    def coste_total_usd(self) -> float:
        return sum(s.coste_usd for s in self.spans)

    @property
    def tokens_totales(self) -> int:
        return sum(s.tokens_input + s.tokens_output for s in self.spans)

    @property
    def n_errores(self) -> int:
        return sum(1 for s in self.spans if not s.ok)

    def resumen(self) -> dict:
        lats = [s.latencia_ms for s in self.spans if s.fin]
        return {
            "trace_id":       self.trace_id,
            "tema":           self.tema[:60],
            "duracion_s":     round(self.duracion_total_s, 2),
            "n_spans":        len(self.spans),
            "n_errores":      self.n_errores,
            "tokens_totales": self.tokens_totales,
            "coste_usd":      round(self.coste_total_usd, 6),
            "latencia_media_ms": round(sum(lats) / len(lats), 1) if lats else 0,
            "latencia_p95_ms":   round(sorted(lats)[int(len(lats)*0.95)], 1) if len(lats) >= 2 else 0,
        }

    def guardar(self) -> Path:
        """Salva la trace completa in un file JSON."""
        ts   = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
        ruta = self.trazas_dir / f"trace_{ts}_{self.trace_id}.json"
        data = {
            **self.resumen(),
            "spans": [
                {
                    "nombre":        s.nombre,
                    "latencia_ms":   round(s.latencia_ms, 1),
                    "tokens_input":  s.tokens_input,
                    "tokens_output": s.tokens_output,
                    "coste_usd":     round(s.coste_usd, 6),
                    "modelo":        s.modelo,
                    "ok":            s.ok,
                    "error":         s.error,
                }
                for s in self.spans
            ],
        }
        ruta.write_text(json.dumps(data, indent=2, ensure_ascii=False))
        return ruta
'''

(OBS_DIR / "tracer.py").write_text(TRACER_PY.strip())
print("✓ observability/tracer.py generato")

# Test del Tracer
import time, importlib, sys
from observability.tracer import Tracer

tracer = Tracer(tema="Test di trace", trazas_dir="/tmp/trazas_test")

# Simula degli span
with tracer.span("planificar") as s:
    time.sleep(0.05)
    s.tokens_input, s.tokens_output = 420, 180

with tracer.span("buscar_web") as s:
    time.sleep(0.12)

with tracer.span("leer_url") as s:
    time.sleep(0.08)

with tracer.span("sintetizar") as s:
    time.sleep(0.15)
    s.tokens_input, s.tokens_output = 14_000, 1_800

r = tracer.resumen()
print("\nRiepilogo trace:")
for k, v in r.items():
    print(f"  {k}: {v}")

ruta = tracer.guardar()
print(f"\n✓ Trace salvata in: {ruta}")

## Cella 3 — Logger strutturato in JSON (`logger.py`)

In [ ]:
LOGGER_PY = '''
"""Logger strutturato in JSON per ResearchAgent."""
import logging
import json
import time
import sys
from pathlib import Path


class JSONFormatter(logging.Formatter):
    """Formatta ogni log come una riga JSON."""

    CAMPOS_EXTRA = ("trace_id", "span", "tokens", "coste_usd", "latencia_ms", "modelo")

    def format(self, record: logging.LogRecord) -> str:
        log = {
            "ts":     time.time(),
            "level":  record.levelname,
            "logger": record.name,
            "msg":    record.getMessage(),
        }
        for campo in self.CAMPOS_EXTRA:
            if hasattr(record, campo):
                log[campo] = getattr(record, campo)
        if record.exc_info:
            log["exc"] = self.formatException(record.exc_info)
        return json.dumps(log, ensure_ascii=False)


def configurar_logger(
    nombre: str = "research_agent",
    nivel: str = "INFO",
    log_file: str | None = None,
) -> logging.Logger:
    """
    Restituisce un logger con output JSON.
    Se viene indicato log_file, scrive anche su disco.
    """
    logger = logging.getLogger(nombre)
    logger.setLevel(getattr(logging, nivel.upper(), logging.INFO))
    logger.handlers.clear()

    fmt = JSONFormatter()

    # Handler verso stdout
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    logger.addHandler(sh)

    # Handler verso file (opzionale)
    if log_file:
        Path(log_file).parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(log_file, encoding="utf-8")
        fh.setFormatter(fmt)
        logger.addHandler(fh)

    return logger


def log_llm_call(
    logger: logging.Logger,
    trace_id: str,
    span: str,
    modelo: str,
    tokens_input: int,
    tokens_output: int,
    latencia_ms: float,
    ok: bool = True,
):
    """Registra una chiamata all'LLM con tutti i campi rilevanti."""
    from observability.cost import calcular_coste
    coste = calcular_coste(tokens_input, tokens_output, modelo)
    extra = {
        "trace_id":    trace_id,
        "span":        span,
        "modelo":      modelo,
        "tokens":      tokens_input + tokens_output,
        "coste_usd":   round(coste, 6),
        "latencia_ms": round(latencia_ms, 1),
    }
    nivel = logging.INFO if ok else logging.ERROR
    msg   = f"llm_call ok={ok} span={span}"
    logger.log(nivel, msg, extra=extra)
'''

(OBS_DIR / "logger.py").write_text(LOGGER_PY.strip())
print("✓ observability/logger.py generato")

# Demo
from observability.logger import configurar_logger, log_llm_call

logger = configurar_logger("demo", nivel="INFO")

log_llm_call(
    logger,
    trace_id="abc123",
    span="planificar",
    modelo="gpt-4o-mini",
    tokens_input=420,
    tokens_output=180,
    latencia_ms=1240.5,
)

# Simula un errore
log_llm_call(
    logger,
    trace_id="abc123",
    span="leer_url",
    modelo="gpt-4o-mini",
    tokens_input=0,
    tokens_output=0,
    latencia_ms=15000.0,
    ok=False,
)

## Cella 4 — Dashboard a riga di comando (`dashboard.py`)

Legge le trace salvate in JSON e mostra un riepilogo cumulativo.

In [ ]:
DASHBOARD_PY = '''
"""Dashboard di osservabilità per ResearchAgent."""
import json
from pathlib import Path


def cargar_trazas(trazas_dir: str = "./trazas") -> list[dict]:
    """Carica tutte le trace salvate come JSON."""
    directorio = Path(trazas_dir)
    if not directorio.exists():
        return []
    trazas = []
    for f in sorted(directorio.glob("trace_*.json")):
        try:
            trazas.append(json.loads(f.read_text()))
        except Exception:
            pass
    return trazas


def imprimir_resumen_traza(traza: dict):
    """Stampa il riepilogo di una singola trace."""
    ok = "✓" if traza["n_errores"] == 0 else f"✗ ({traza[\'n_errores\']} errori)"
    print(f"  [{traza[\'trace_id\']}] {ok}  {traza[\'duracion_s\']}s  "
          f"${traza[\'coste_usd\']:.5f}  {traza[\'tokens_totales\']}tok  "
          f"{traza[\'tema\'][:50]}")


def dashboard(trazas_dir: str = "./trazas", ultimas_n: int = 20):
    """Mostra la dashboard delle ultime N trace."""
    trazas = cargar_trazas(trazas_dir)[-ultimas_n:]

    print("=" * 70)
    print(f" ResearchAgent — Dashboard di osservabilità ({len(trazas)} trace)")
    print("=" * 70)

    if not trazas:
        print(" Nessuna trace disponibile.")
        return

    # Metriche cumulative
    coste_total      = sum(t["coste_usd"] for t in trazas)
    tokens_total     = sum(t["tokens_totales"] for t in trazas)
    duraciones       = [t["duracion_s"] for t in trazas]
    tasa_exito       = sum(1 for t in trazas if t["n_errores"] == 0) / len(trazas)
    lats_sorted      = sorted(duraciones)
    p95_idx          = max(0, int(len(lats_sorted) * 0.95) - 1)

    print(f"\n Riepilogo globale")
    print(f"  Costo totale:          ${coste_total:.4f}")
    print(f"  Token totali:           {tokens_total:,}")
    print(f"  Durata media:           {sum(duraciones)/len(duraciones):.1f}s")
    print(f"  Durata p95:             {lats_sorted[p95_idx]:.1f}s")
    print(f"  Tasso di successo:      {tasa_exito*100:.0f}%")
    if len(trazas) > 1:
        coste_medio = coste_total / len(trazas)
        print(f"  Costo medio/report:   ${coste_medio:.5f}")

    # Top span per costo
    todos_spans: list[dict] = []
    for t in trazas:
        todos_spans.extend(t.get("spans", []))

    if todos_spans:
        print(f"\n Top span per costo accumulato")
        por_nombre: dict[str, dict] = {}
        for s in todos_spans:
            n = s["nombre"]
            if n not in por_nombre:
                por_nombre[n] = {"coste": 0.0, "tokens": 0, "n": 0, "errores": 0}
            por_nombre[n]["coste"]   += s["coste_usd"]
            por_nombre[n]["tokens"]  += s["tokens_input"] + s["tokens_output"]
            por_nombre[n]["n"]       += 1
            por_nombre[n]["errores"] += 0 if s["ok"] else 1

        for nombre, m in sorted(por_nombre.items(),
                                 key=lambda x: x[1]["coste"], reverse=True)[:5]:
            tasa_err = m["errores"] / m["n"] * 100 if m["n"] else 0
            print(f"  {nombre:<25} ${m[\'coste\']:.5f}  "
                  f"{m[\'tokens\']:>8,} tok  "
                  f"{m[\'n\']:>3}x  "
                  f"err={tasa_err:.0f}%")

    # Ultime trace
    print(f"\n Ultime {min(5, len(trazas))} trace")
    for t in trazas[-5:]:
        imprimir_resumen_traza(t)

    print("=" * 70)
'''

(OBS_DIR / "dashboard.py").write_text(DASHBOARD_PY.strip())
print("✓ observability/dashboard.py generato")

# Demo con dati simulati
from observability.dashboard import dashboard

# Genera varie trace di prova
import random, time
TRAZAS_TEST = "/tmp/trazas_dashboard_demo"

for i in range(6):
    t = Tracer(tema=f"Tema di prova numero {i+1} sull'IA nel settore X",
               trazas_dir=TRAZAS_TEST)
    with t.span("planificar") as s:
        time.sleep(random.uniform(0.01, 0.03))
        s.tokens_input, s.tokens_output = random.randint(300, 600), random.randint(100, 300)
    for j in range(random.randint(2, 4)):
        with t.span("ejecutar_subtarea") as s:
            time.sleep(random.uniform(0.02, 0.08))
            s.tokens_input  = random.randint(1500, 4000)
            s.tokens_output = random.randint(200, 600)
            s.ok = random.random() > 0.15  # 15% di fallimenti simulati
    with t.span("sintetizar") as s:
        time.sleep(random.uniform(0.05, 0.12))
        s.tokens_input  = random.randint(10_000, 20_000)
        s.tokens_output = random.randint(1_000, 2_500)
    t.guardar()

print()
dashboard(trazas_dir=TRAZAS_TEST)

## Cella 5 — Strumentare ResearchAgent

Aggiungiamo le trace al `ResearchAgent` reale senza riscrivere la logica.

In [ ]:
AGENT_INSTRUMENTED_PY = '''
"""ResearchAgent con osservabilità integrata."""
import time
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field
from typing import Callable

from agent.planner     import crear_plan
from agent.executor    import ejecutar_subtarea
from agent.synthesizer import generar_informe
from observability.tracer import Tracer
from observability.logger import configurar_logger, log_llm_call


@dataclass
class ResultadoInvestigacion:
    tema:          str
    plan:          list[dict]
    resultados:    list[dict]
    informe:       str
    duracion_s:    float
    n_fuentes:     int
    ruta_informe:  str = ""
    trace_id:      str = ""
    coste_usd:     float = 0.0


class ResearchAgent:
    """
    ResearchAgent con osservabilità integrata.
    Emette trace a ogni fase e salva su disco il report + la trace.
    """

    def __init__(
        self,
        informes_dir: str = "./informes",
        trazas_dir:   str = "./trazas",
        log_file:     str | None = None,
        verbose:      bool = True,
    ):
        self.informes_dir = Path(informes_dir)
        self.trazas_dir   = Path(trazas_dir)
        self.informes_dir.mkdir(parents=True, exist_ok=True)
        self.trazas_dir.mkdir(parents=True, exist_ok=True)
        self.verbose = verbose
        self.logger  = configurar_logger(
            "research_agent",
            nivel="INFO",
            log_file=log_file,
        )

    def _log(self, msg: str):
        if self.verbose:
            print(msg)

    def investigar(
        self,
        tema: str,
        on_progreso: Callable[[int, int, str], None] | None = None,
    ) -> ResultadoInvestigacion:
        tracer = Tracer(tema=tema, trazas_dir=str(self.trazas_dir))
        self._log(f"\n🔍 [{tracer.trace_id}] Avvio: {tema[:60]}\n")

        # ── Fase 1: Pianificazione ────────────────────────────────────────────
        self._log("📋 Fase 1/3 — Pianificazione...")
        with tracer.span("planificar") as s:
            plan = crear_plan(tema)
            s.tokens_input  = 500   # stima; in produzione usa tiktoken
            s.tokens_output = len(plan) * 80
        self._log(f"   {len(plan)} sottotask")

        # ── Fase 2: Esecuzione ───────────────────────────────────────────────
        self._log("\n🔎 Fase 2/3 — Ricerca in corso...")
        resultados = []
        for i, subtarea in enumerate(plan, 1):
            with tracer.span(f"subtarea_{i}") as s:
                resultado = ejecutar_subtarea(subtarea)
                # In produzione, ejecutar_subtarea dovrebbe restituire i token reali
                s.tokens_input  = len(subtarea["descripcion"]) * 4
                s.tokens_output = len(resultado["hallazgo"]) // 4
            resultados.append(resultado)

            if on_progreso:
                on_progreso(i, len(plan), subtarea["descripcion"])

            self._log(f"   {i}/{len(plan)} — {resultado[\'n_pasos\']} passi, "
                      f"{len(resultado[\'fuentes\'])} fonti")

        # ── Fase 3: Sintesi ───────────────────────────────────────────────────
        self._log("\n✍️  Fase 3/3 — Sintesi...")
        with tracer.span("sintetizar") as s:
            informe      = generar_informe(tema, resultados)
            s.tokens_input  = sum(len(r["hallazgo"]) for r in resultados) // 4
            s.tokens_output = len(informe) // 4

        # ── Salvataggio del report ────────────────────────────────────────────
        duracion_s = tracer.duracion_total_s
        n_fuentes  = sum(len(r["fuentes"]) for r in resultados)
        fecha      = datetime.now().strftime("%Y%m%d_%H%M")
        slug       = tema[:40].replace(" ", "_").replace("/", "-")
        ruta       = self.informes_dir / f"{fecha}_{slug}.md"

        cabecera = (
            f"---\ntema: {tema}\ntrace_id: {tracer.trace_id}\n"
            f"coste_usd: {tracer.coste_total_usd:.6f}\n"
            f"duracion_s: {duracion_s:.1f}\nn_fuentes: {n_fuentes}\n---\n\n"
        )
        ruta.write_text(cabecera + informe, encoding="utf-8")

        ruta_traza = tracer.guardar()
        res = tracer.resumen()

        self._log(f"\n✅ Fatto [{tracer.trace_id}]  "
                  f"{duracion_s:.1f}s  "
                  f"${tracer.coste_total_usd:.5f}  "
                  f"{tracer.tokens_totales:,} token")

        return ResultadoInvestigacion(
            tema=tema, plan=plan, resultados=resultados,
            informe=informe, duracion_s=duracion_s, n_fuentes=n_fuentes,
            ruta_informe=str(ruta),
            trace_id=tracer.trace_id,
            coste_usd=tracer.coste_total_usd,
        )
'''

# Sovrascrive agent.py con la versione strumentata
(PROJECT_DIR / "agent" / "agent.py").write_text(AGENT_INSTRUMENTED_PY.strip())
print("✓ agent/agent.py aggiornato con osservabilità")

## Cella 6 — Esecuzione completa con trace

Esegue l'agente strumentato e mostra la dashboard al termine.

In [ ]:
import importlib
import agent.agent
importlib.reload(agent.agent)

from agent.agent import ResearchAgent
from observability.dashboard import dashboard
from IPython.display import Markdown, display

TRAZAS_DIR   = str(PROJECT_DIR / "trazas")
INFORMES_DIR = str(PROJECT_DIR / "informes")

agente = ResearchAgent(
    informes_dir=INFORMES_DIR,
    trazas_dir=TRAZAS_DIR,
    verbose=True,
)

resultado = agente.investigar(
    "Digitalizzazione del settore forestale in Galizia: tecnologie emergenti"
)

print(f"\nTrace ID:  {resultado.trace_id}")
print(f"Costo:     ${resultado.coste_usd:.5f}")
print(f"Durata:    {resultado.duracion_s:.1f}s")
print(f"Fonti:     {resultado.n_fuentes}")

In [ ]:
# Dashboard con la trace reale
dashboard(trazas_dir=TRAZAS_DIR)

print()
display(Markdown(resultado.informe))

## Cella 7 — Analisi delle trace con Python

Le trace in JSON permettono di rispondere a domande di business senza una piattaforma.

In [ ]:
import json
from pathlib import Path

trazas_dir = Path(TRAZAS_DIR)
trazas = [
    json.loads(f.read_text())
    for f in sorted(trazas_dir.glob("trace_*.json"))
]

if not trazas:
    print("Nessuna trace — esegui prima la cella 6")
else:
    print(f"Totale trace analizzate: {len(trazas)}\n")

    # 1. Quanto costa in media un report?
    coste_medio = sum(t["coste_usd"] for t in trazas) / len(trazas)
    print(f"Costo medio per report:     ${coste_medio:.5f}")

    # 2. Quale fase consuma più token?
    tokens_por_span: dict[str, int] = {}
    for t in trazas:
        for s in t.get("spans", []):
            n = s["nombre"]
            tokens_por_span[n] = tokens_por_span.get(n, 0) + s["tokens_input"] + s["tokens_output"]

    print("\nToken per fase (accumulati):")
    for nombre, tok in sorted(tokens_por_span.items(), key=lambda x: -x[1]):
        print(f"  {nombre:<25} {tok:>10,} token")

    # 3. Proiezione del costo se il sistema elabora 50 report/giorno?
    coste_50_dia = coste_medio * 50
    print(f"\nProiezione 50 report/giorno: ${coste_50_dia:.3f}/giorno → ${coste_50_dia*30:.2f}/mese")

    # 4. Tasso di successo
    exitosos = sum(1 for t in trazas if t["n_errores"] == 0)
    print(f"Tasso di successo:           {exitosos}/{len(trazas)} ({exitosos/len(trazas)*100:.0f}%)")

## Sfida — Aggiungere token reali dalla risposta di OpenAI

L'attuale `executor.py` non restituisce i token reali di ogni chiamata: gli span di `agent.py`
usano stime basate sulla lunghezza del testo (`len(texto) // 4`).

Modifica `ejecutar_subtarea()` in `executor.py` affinché restituisca anche i token reali:

```python
# Risultato attuale
{
    "subtarea_id": 1,
    "descripcion": "...",
    "hallazgo": "...",
    "fuentes": [...],
    "n_pasos": 3,
}

# Risultato con token reali
{
    "subtarea_id": 1,
    ...
    "tokens_input_total":  4820,   # somma di tutti i prompt_tokens del ciclo ReAct
    "tokens_output_total": 610,    # somma di tutti i completion_tokens
}
```

Suggerimento: nel ciclo ReAct di `executor.py`, ogni chiamata a `client.chat.completions.create()`
restituisce `resp.usage.prompt_tokens` e `resp.usage.completion_tokens`.
Accumulali in contatori e aggiungili al dizionario del risultato.

Poi, in `agent.py`, usa questi valori reali al posto delle stime:
```python
s.tokens_input  = resultado["tokens_input_total"]
s.tokens_output = resultado["tokens_output_total"]
```

## Riepilogo della settimana 17

| Componente | File | Cosa offre |
|-----------|---------|------------|
| Calcolatrice di costi | `observability/cost.py` | Costo esatto per chiamata, proiezione mensile |
| Tracer + Span | `observability/tracer.py` | Latenza, token ed errori per fase |
| Logger JSON | `observability/logger.py` | Log filtrabili in produzione |
| Dashboard a riga di comando | `observability/dashboard.py` | Riepilogo di più esecuzioni |
| Agent strumentato | `agent/agent.py` | Trace senza contaminare la logica |
| Analisi delle trace | Cella 7 | Risponde a domande di business |
| Token reali | Sfida | Precisione nel tracking dei costi |

La settimana 18 collega questo sistema di trace a **Langfuse** (open source) per
avere una dashboard visiva, confrontare versioni di prompt e configurare alert.